# 03 — Feature Engineering

## Import Libraries

In [1]:
import sys
sys.path.insert(0, "..")  # so `from src import features` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import features as feat

plt.style.use('default')
sns.set_theme(style="whitegrid")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")
print(f"src.features loaded - {len(feat.MODEL_FEATURES)} canonical model features defined.")

All libraries imported successfully.
src.features loaded - 41 canonical model features defined.


## Load cleaned dataset

We load `hotel_bookings_clean.csv` (post-dedup, pre-EDA-bucket columns) rather than the
EDA-enriched file, since this notebook now owns the *canonical* feature engineering via
`src/features.py` and we don't want the EDA notebook's exploratory bucket columns to be
confused with the modeling ones.

In [2]:
DATA_PATH = "../data/hotel_bookings_clean.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded ../data/hotel_bookings_clean.csv: 87,396 rows, 32 columns


In [3]:
required_columns = [
    "is_canceled", "lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "previous_cancellations", "booking_changes",
    "total_of_special_requests", "deposit_type", "customer_type", "market_segment",
    "hotel", "arrival_date_month"
]

missing_required_columns = [col for col in required_columns if col not in df.columns]

if not missing_required_columns:
    print("All required columns are available.")
else:
    print("Missing columns:", missing_required_columns)
    raise ValueError("Missing required columns: " + str(missing_required_columns))

All required columns are available.


## Apply Canonical Feature Engineering

A single call to `feat.engineer_all_features()` produces every engineered column used by
both the notebooks and the app - stay-duration features, lead-time/ADR buckets, guest
behaviour signals, revenue proxies, and room-type comparison flags (kept for EDA narrative,
excluded from modeling - see Leakage Review below).

In [4]:
rows_before = df.shape[0]
cols_before = df.shape[1]

df = feat.engineer_all_features(df)

print(f"Rows: {rows_before:,} -> {df.shape[0]:,} (engineering must not change row count)")
print(f"Columns: {cols_before} -> {df.shape[1]}")
assert df.shape[0] == rows_before, "Row count changed during feature engineering"

Rows: 87,396 -> 87,396 (engineering must not change row count)
Columns: 32 -> 58


### Validate engineered features

In [5]:
existing_engineered_features = [
    "total_nights", "estimated_booking_value", "lead_time_bucket", "adr_bucket"
]
for col in existing_engineered_features:
    print(f"{col}: {col in df.columns}")

total_nights: True
estimated_booking_value: True
lead_time_bucket: True
adr_bucket: True


In [6]:
# Validate total_nights
total_nights_check = df["total_nights"] == df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
print(f"Correct total_nights rows: {total_nights_check.sum():,}")
print(f"Incorrect total_nights rows: {(~total_nights_check).sum():,}")
assert (~total_nights_check).sum() == 0

Correct total_nights rows: 87,396
Incorrect total_nights rows: 0


In [7]:
# Validate estimated_booking_value
estimated_value_check = np.isclose(df["estimated_booking_value"], df["adr"] * df["total_nights"])
print(f"Correct estimated_booking_value rows: {estimated_value_check.sum():,}")
print(f"Incorrect estimated_booking_value rows: {(~estimated_value_check).sum():,}")
assert (~estimated_value_check).sum() == 0

Correct estimated_booking_value rows: 87,396
Incorrect estimated_booking_value rows: 0


In [8]:
print("Missing values in engineered features:")
df[existing_engineered_features].isna().sum()

Missing values in engineered features:


total_nights               0
estimated_booking_value    0
lead_time_bucket           0
adr_bucket                 0
dtype: int64

In [9]:
print("lead_time_bucket distribution:")
print(df["lead_time_bucket"].value_counts(dropna=False))
print()
print("adr_bucket distribution:")
print(df["adr_bucket"].value_counts(dropna=False))

lead_time_bucket distribution:
lead_time_bucket
0-30 Days       34644
31-90 Days      22744
91-180 Days     18243
181-365 Days    11200
365+ Days         565
Name: count, dtype: int64

adr_bucket distribution:
adr_bucket
51-100         35793
101-150        26632
151-200        10395
0-50            9857
201-300         4457
300+             261
Invalid ADR        1
Name: count, dtype: int64


**Bug-regression check.** This is the single most important cell in this notebook. It
directly verifies that the bucket labels actually produced by the data match the ordinal
category lists that will be handed to `OrdinalEncoder` in notebook 04. If this assertion
ever fails, it means someone edited a bucket function or a category list in only one place
- exactly the failure mode that broke the original project.

In [10]:
for col, order in zip(feat.ORDINAL_FEATURES, feat.ORDINAL_CATEGORIES):
    produced = set(df[col].dropna().unique())
    expected = set(order)
    unmatched = produced - expected
    assert not unmatched, col + ": labels " + str(unmatched) + " not found in ordinal category list"
    print(f"OK  {col}: all {len(produced)} produced labels are present in the ordinal category list.")

OK  lead_time_bucket: all 5 produced labels are present in the ordinal category list.
OK  adr_bucket: all 7 produced labels are present in the ordinal category list.
OK  stay_length_category: all 5 produced labels are present in the ordinal category list.


## Leakage Review

In [11]:
leakage_cols = feat.LEAKAGE_COLUMNS
for col in leakage_cols:
    print(f"{col}: present in df = {col in df.columns}")

print()
print("Additional columns excluded as leakage-risk (not in original project):")
for col in ["room_type_changed", "reserved_room_type", "assigned_room_type"]:
    print(f"  {col}: present in df = {col in df.columns} (will be EXCLUDED from model features)")

reservation_status: present in df = True
reservation_status_date: present in df = True

Additional columns excluded as leakage-risk (not in original project):
  room_type_changed: present in df = True (will be EXCLUDED from model features)
  reserved_room_type: present in df = True (will be EXCLUDED from model features)
  assigned_room_type: present in df = True (will be EXCLUDED from model features)


## Feature Selection Setup

In [12]:
candidate_features = feat.MODEL_FEATURES.copy()
print(f"Number of candidate (model) features: {len(candidate_features)}")
candidate_features

Number of candidate (model) features: 41


['lead_time',
 'adr',
 'total_nights',
 'estimated_booking_value',
 'total_of_special_requests',
 'booking_changes',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'previous_total_bookings',
 'previous_cancellation_rate',
 'required_car_parking_spaces',
 'arrival_month_number',
 'adults',
 'children',
 'babies',
 'hotel',
 'market_segment',
 'distribution_channel',
 'deposit_type',
 'customer_type',
 'arrival_season',
 'lead_time_bucket',
 'adr_bucket',
 'stay_length_category',
 'is_repeated_guest',
 'has_invalid_adr',
 'is_zero_night_stay',
 'is_weekend_stay',
 'is_last_minute_booking',
 'is_long_lead_booking',
 'is_peak_season',
 'has_previous_cancellation',
 'has_previous_successful_booking',
 'has_booking_changes',
 'has_special_requests',
 'is_waitlisted',
 'repeat_guest_with_deposit',
 'repeat_guest_non_refund',
 'is_zero_adr',
 'is_high_adr',
 'is_high_value_booking']

In [13]:
X = df[candidate_features].copy()
y = df["is_canceled"].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
assert X.shape[0] == y.shape[0]

X shape: (87396, 41)
y shape: (87396,)


## Encode & Scale - preview only

The actual fit/transform happens in notebook 04 on the train split, to avoid any chance of
fitting on data that includes the test set. This section just confirms the preprocessing
*architecture* works mechanically on the full feature matrix before we move to train/test
splitting.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler

preprocessor_preview = ColumnTransformer(
    transformers=[
        ("numeric", RobustScaler(), feat.NUMERIC_FEATURES),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), feat.NOMINAL_FEATURES),
        (
            "ordinal",
            OrdinalEncoder(
                categories=feat.ORDINAL_CATEGORIES,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            feat.ORDINAL_FEATURES,
        ),
        ("binary", "passthrough", feat.BINARY_FEATURES),
    ],
    remainder="drop",
)

X_preprocessed_preview = preprocessor_preview.fit_transform(X)
print(f"Original feature matrix shape: {X.shape}")
print(f"Preprocessed feature matrix shape: {X_preprocessed_preview.shape}")

Original feature matrix shape: (87396, 41)
Preprocessed feature matrix shape: (87396, 61)


**Bug-regression check (architecture-level).** Confirm the ordinal columns actually
produced real encoded values, not silently all -1.

In [15]:
ordinal_start = len(feat.NUMERIC_FEATURES) + len(
    preprocessor_preview.named_transformers_["nominal"].get_feature_names_out(feat.NOMINAL_FEATURES)
)
ordinal_block = X_preprocessed_preview[:, ordinal_start: ordinal_start + len(feat.ORDINAL_FEATURES)]

for i, col in enumerate(feat.ORDINAL_FEATURES):
    n_unknown = (ordinal_block[:, i] == -1).sum()
    pct_unknown = n_unknown / len(ordinal_block) * 100
    print(f"{col}: {n_unknown:,} rows encoded as unknown (-1) -- {pct_unknown:.2f}% of data")
    assert pct_unknown < 1.0, col + " has " + str(round(pct_unknown,1)) + "% unknown-encoded rows"

print()
print("All ordinal features encode correctly (well under 1% unknown).")

lead_time_bucket: 0 rows encoded as unknown (-1) -- 0.00% of data
adr_bucket: 0 rows encoded as unknown (-1) -- 0.00% of data
stay_length_category: 0 rows encoded as unknown (-1) -- 0.00% of data

All ordinal features encode correctly (well under 1% unknown).


### Featured Engineered Dataset

In [16]:
OUTPUT_PATH = "../data/hotel_bookings_feature_engineered.csv"
df.to_csv(OUTPUT_PATH, index=False)

print(f"Feature-engineered dataset saved to: {OUTPUT_PATH}")
print(f"Rows saved: {df.shape[0]:,}")
print(f"Columns saved: {df.shape[1]:,}")

Feature-engineered dataset saved to: ../data/hotel_bookings_feature_engineered.csv
Rows saved: 87,396
Columns saved: 58
